# Solution 03: Fine-Tuning GLiNER

In [1]:
from gliner import GLiNER
import json, os, re
from collections import Counter

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "training_examples.json")) as f:
    train_data = json.load(f)

print(f"✓ Loaded {len(train_data)} training examples")

✓ Loaded 20 training examples


## Part 1: Validate Training Data

In [2]:
def validate_training_data(examples):
    """Validate GLiNER training format — returns list of error messages."""
    errors = []
    for i, ex in enumerate(examples):
        if "tokenized_text" not in ex:
            errors.append(f"Example {i}: missing 'tokenized_text'")
            continue
        if "ner" not in ex:
            errors.append(f"Example {i}: missing 'ner'")
            continue
        n = len(ex["tokenized_text"])
        for span in ex["ner"]:
            if len(span) != 3:
                errors.append(f"Example {i}: span {span} must have 3 elements")
                continue
            start, end, label = span
            if not isinstance(start, int) or not isinstance(end, int):
                errors.append(f"Example {i}: span indices must be ints, got {span}")
            elif start < 0 or end >= n:
                errors.append(f"Example {i}: span [{start},{end}] out of bounds (len={n})")
            elif start > end:
                errors.append(f"Example {i}: span start {start} > end {end}")
            if not isinstance(label, str) or not label:
                errors.append(f"Example {i}: invalid label {label!r}")
    return errors


errors = validate_training_data(train_data)
assert len(errors) == 0, "\n".join(errors)
print(f"✓ All {len(train_data)} examples are valid")

bad_example = {"tokenized_text": ["Apple", "Inc"], "ner": [[0, 5, "organization"]]}
bad_errors = validate_training_data([bad_example])
assert len(bad_errors) > 0
print(f"✓ Bad example correctly detected: {bad_errors[0]}")

label_counts = Counter(span[2] for ex in train_data for span in ex["ner"])
print("\nLabel distribution:")
for label, count in label_counts.most_common():
    print(f"  {label:20} {count}")

✓ All 20 examples are valid
✓ Bad example correctly detected: Example 0: span [0,5] out of bounds (len=2)

Label distribution:
  person               22
  organization         19
  date                 19
  location             13
  product              1
  event                1


## Part 2: Character-Level Converter

In [3]:
def from_char_annotations(text, char_entities):
    """Convert character-level entity spans to GLiNER training format."""
    # Tokenize by finding non-whitespace runs
    token_matches = list(re.finditer(r'\S+', text))
    tokens = [m.group() for m in token_matches]
    token_spans = [(m.start(), m.end()) for m in token_matches]

    ner = []
    for ent in char_entities:
        cs, ce = ent["start"], ent["end"]
        # Find tokens whose span is contained within the entity char span
        entity_toks = [
            i for i, (ts, te) in enumerate(token_spans)
            if ts >= cs and te <= ce
        ]
        if entity_toks:
            ner.append([entity_toks[0], entity_toks[-1], ent["label"]])

    return {"tokenized_text": tokens, "ner": ner}


test_text = "Elon Musk founded SpaceX in Hawthorne California"
char_entities = [
    {"text": "Elon Musk",            "label": "person",       "start": 0,  "end": 9},
    {"text": "SpaceX",               "label": "organization", "start": 18, "end": 24},
    {"text": "Hawthorne California", "label": "location",     "start": 28, "end": 48},
]
result = from_char_annotations(test_text, char_entities)

assert len(result["ner"]) == 3
tokens = result["tokenized_text"]
for span, expected in zip(result["ner"], char_entities):
    start, end, label = span
    assert label == expected["label"]
    assert " ".join(tokens[start:end + 1]) == expected["text"]
assert validate_training_data([result]) == []

print("✓ from_char_annotations works correctly")
for start, end, label in result["ner"]:
    print(f"  [{label:12}] [{start},{end}] = {' '.join(tokens[start:end+1])!r}")

✓ from_char_annotations works correctly
  [person      ] [0,1] = 'Elon Musk'
  [organization] [3,3] = 'SpaceX'
  [location    ] [5,6] = 'Hawthorne California'


## Part 3: Fine-Tune the Model

In [4]:
output_dir = os.path.join(os.getcwd(), "..", "trained_model")

model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")

# Split 16 train / 4 eval
train_split = train_data[:16]
eval_split  = train_data[16:]

trainer = model.train_model(
    train_split,
    eval_split,
    output_dir=output_dir,
    max_steps=200,
    learning_rate=1e-5,
    others_lr=3e-5,
    per_device_train_batch_size=4,
    negatives=1.5,
    warmup_ratio=0.1,
    save_steps=200,
    save_total_limit=1,
)

trainer.model.save_pretrained(output_dir)
print(f"✓ Training complete")
assert os.path.isdir(output_dir)
saved_files = os.listdir(output_dir)
assert any(f.endswith(".pt") or f.endswith(".bin") or f == "config.json" for f in saved_files)
print(f"✓ Model saved to {output_dir}, files: {saved_files}")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:1141: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(**trainer_kwargs)
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50283}.


Step,Training Loss
10,15.146400
20,4.444000
30,0.102800
40,0.000400
50,0.000700
60,0.000000
70,0.000100
80,0.000000
90,0.000000
100,0.000000


✓ Training complete
✓ Model saved to /home/biomike/BioMike/nlp-puzzles/modules/15_ner_gliner/solutions/../trained_model, files: ['gliner_config.json', 'tokenizer.json', 'tokenizer_config.json', 'checkpoint-200', 'special_tokens_map.json', 'pytorch_model.bin']


## Part 4: Compare Base vs. Fine-Tuned

In [5]:
test_sentences = [
    "Howie Liu founded Airtable in San Francisco in 2012",
    "Dylan Field co-founded Figma in San Francisco in 2012",
    "Stewart Butterfield launched Slack in Vancouver in 2013",
]
labels = ["person", "organization", "location", "date"]

finetuned_model = GLiNER.from_pretrained(output_dir)
base_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")

assert hasattr(finetuned_model, 'predict_entities')

print("Comparison: Base vs. Fine-Tuned\n")
all_ft_results = []
for sentence in test_sentences:
    base_ents = base_model.predict_entities(sentence, labels, threshold=0.2)
    ft_ents = finetuned_model.predict_entities(sentence, labels, threshold=0.2)
    all_ft_results.append(ft_ents)
    print(f"  {sentence}")
    print(f"    Base ({len(base_ents)}): {[(e['text'], e['label']) for e in base_ents]}")
    print(f"    FT   ({len(ft_ents)}): {[(e['text'], e['label']) for e in ft_ents]}")
    print()

total = sum(len(r) for r in all_ft_results)
assert total > 0, "Fine-tuned model found no entities"
print(f"✓ Fine-tuned model found {total} entities across test sentences")

/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/gliner/model.py:447: UserWarning: Resizing embeddings is not supported for bi-encoder models.
  instance.resize_embeddings()


/home/biomike/BioMike/nlp-puzzles/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Comparison: Base vs. Fine-Tuned

  Howie Liu founded Airtable in San Francisco in 2012
    Base (4): [('Howie Liu', 'person'), ('Airtable', 'organization'), ('San Francisco', 'location'), ('2012', 'date')]
    FT   (4): [('Howie Liu', 'person'), ('Airtable', 'organization'), ('San Francisco', 'location'), ('2012', 'date')]

  Dylan Field co-founded Figma in San Francisco in 2012
    Base (4): [('Dylan Field', 'person'), ('Figma', 'organization'), ('San Francisco', 'location'), ('2012', 'date')]
    FT   (4): [('Dylan Field', 'person'), ('Figma', 'organization'), ('San Francisco', 'location'), ('2012', 'date')]

  Stewart Butterfield launched Slack in Vancouver in 2013
    Base (4): [('Stewart Butterfield', 'person'), ('Slack', 'organization'), ('Vancouver', 'location'), ('2013', 'date')]
    FT   (4): [('Stewart Butterfield', 'person'), ('Slack', 'organization'), ('Vancouver', 'location'), ('2013', 'date')]

✓ Fine-tuned model found 12 entities across test sentences
